# 🎬 Cinematic Engine V17 — Enterprise AI Platform

**One-click setup for Google Colab T4 GPU**

### الخطوات بالترتيب
1. `Runtime → Change runtime type → T4 GPU → Save`
2. شغّل **Cell 1** — تحميل المشروع وتثبيت الـ packages (3-5 دقائق)
3. شغّل **Cell 2** — تحميل الـ Engine
4. شغّل **Cell 3** — تشغيل الـ Bridge
5. افتح `cinematic_engine_v17_2_final.html` في المتصفح
6. اضغط **WS: OFF** في الـ Dashboard عشان تتصل

> **ملاحظة:** لو بتفتح الـ Dashboard من موبايل أو جهاز تاني، فعّل `USE_NGROK = True` في Cell 3


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1 — تحميل المشروع وتثبيت الـ Packages            ║
# ║  شغّله مرة واحدة في كل session                          ║
# ╚══════════════════════════════════════════════════════════╝

import subprocess, os, sys, torch

# ── GPU Check ────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError(
        '\n╔══════════════════════════════════════════╗'
        '\n║  NO GPU — Runtime → Change → T4 GPU     ║'
        '\n╚══════════════════════════════════════════╝'
    )
gpu  = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✓ GPU: {gpu} | VRAM: {vram:.1f} GB')
rec = 'FLUX DEV' if vram>=20 else 'FLUX SCHNELL' if vram>=14 else 'SDXL'
print(f'✓ Recommended model: {rec}')

# ── تحميل المشروع من GitHub ──────────────────────────────────
REPO_URL  = 'https://github.com/michelluke1984-cyber/cinematic-engine'
REPO_DIR  = '/content/cinematic-engine'

if os.path.exists(REPO_DIR):
    print('\n↷ المشروع موجود بالفعل — بيتحدّث...')
    os.system(f'cd {REPO_DIR} && git pull -q')
else:
    print('\n⬇ تحميل المشروع من GitHub...')
    os.system(f'git clone -q {REPO_URL} {REPO_DIR}')

# التأكد إن الملفات المهمة موجودة
required = [
    f'{REPO_DIR}/cinematic_engine_v16_pro.py',
    f'{REPO_DIR}/cev17_backend.py',
]
all_ok = True
for f in required:
    if os.path.exists(f):
        print(f'  ✓ {os.path.basename(f)}')
    else:
        print(f'  ✗ {os.path.basename(f)} — مش موجود!')
        all_ok = False

if not all_ok:
    raise FileNotFoundError('ملفات ناقصة في الـ repository — اتأكد إنها اترفعت على GitHub')

# ── نقل للـ working directory ─────────────────────────────────
os.chdir(REPO_DIR)
print(f'\n✓ Working directory: {os.getcwd()}')

# ── تثبيت الـ packages ───────────────────────────────────────
def run(cmd, label):
    print(f'  Installing {label}…', end=' ', flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 and r.stderr.strip():
        print(f'⚠  {r.stderr.strip()[:80]}')
    else:
        print('✓')
    return r.returncode

pkgs = [
    ('pip install -q diffusers transformers accelerate xformers triton einops', 'diffusers + transformers'),
    ('pip install -q gfpgan realesrgan tqdm sentencepiece nltk',               'GFPGAN + NLTK'),
    ('pip install -q controlnet-aux open-clip-torch',                          'ControlNet'),
    ('pip install -q "gradio>=4.0.0"',                                         'Gradio'),
    ('pip install -q bitsandbytes Pillow',                                     'bitsandbytes + Pillow'),
    ('pip install -q insightface onnxruntime-gpu',                             'InsightFace (FaceID)'),
    ('pip install -q websockets',                                              'WebSocket bridge'),
    ('pip install -q pyngrok',                                                 'ngrok tunnel'),
]
print()
for cmd, label in pkgs:
    run(cmd, label)

# ── NLTK data ─────────────────────────────────────────────────
import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
print('  ✓ NLTK data')

# ── تحميل الـ models ─────────────────────────────────────────
print('\n⬇ تحميل الـ model weights...')
os.makedirs('ip_adapter_sdxl/image_encoder', exist_ok=True)
downloads = {
    'GFPGANv1.3.pth':           'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth',
    'realesr-general-x4v3.pth': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth',
    'ip_adapter_sdxl.bin':      'https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/ip-adapter_sdxl.bin',
}
for fname, url in downloads.items():
    if os.path.exists(fname):
        print(f'  ↷ {fname} موجود بالفعل ({os.path.getsize(fname)/1e6:.0f}MB)')
    else:
        print(f'  ⬇ {fname}…', end=' ', flush=True)
        ret = subprocess.run(f'wget -q -O {fname} {url}', shell=True)
        if ret.returncode == 0:
            print(f'✓ ({os.path.getsize(fname)/1e6:.0f}MB)')
        else:
            print('⚠ فشل التحميل — هيحاول يكمل')

print('\n' + '═'*55)
print('  ✓ CELL 1 مكتمل — شغّل Cell 2 دلوقتي')
print('═'*55)

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 2 — تحميل الـ V16 Engine                          ║
# ╚══════════════════════════════════════════════════════════╝

import importlib.util, sys, os

# ── التأكد من الـ working directory ───────────────────────────
REPO_DIR = '/content/cinematic-engine'
if os.getcwd() != REPO_DIR:
    os.chdir(REPO_DIR)
    print(f'✓ Changed to: {REPO_DIR}')

ENGINE_PATH = os.path.join(REPO_DIR, 'cinematic_engine_v16_pro.py')

if not os.path.exists(ENGINE_PATH):
    raise FileNotFoundError(
        f'\n✗ الملف مش موجود: {ENGINE_PATH}'
        f'\n  تأكد إن Cell 1 اشتغل أولاً'
    )

print(f'✓ Engine file found: {ENGINE_PATH}')
print('[ENGINE] Loading CinematicEngineV16…')

# ── تحميل الـ V16 كـ module ────────────────────────────────────
spec  = importlib.util.spec_from_file_location('cev16', ENGINE_PATH)
cev16 = importlib.util.module_from_spec(spec)

# تسجيله في sys.modules أولاً عشان الـ imports الداخلية تشتغل
sys.modules['cev16'] = cev16
spec.loader.exec_module(cev16)

# نقل كل الـ classes والـ functions للـ global scope
EXPORTS = [
    'CinematicEngineV16', 'ModelMode', 'SpeedMode', 'PhysicsMode',
    'GenerationConfig', 'Config', 'VRAMManager', 'PipelineManager',
    'CinematicLogger', 'SmartCache', 'logger',
]
imported = []
for name in EXPORTS:
    obj = getattr(cev16, name, None)
    if obj is not None:
        globals()[name] = obj
        sys.modules['__main__'].__dict__[name] = obj
        imported.append(name)

# تصدير الباقي
for k, v in vars(cev16).items():
    if not k.startswith('_') and k not in imported:
        globals()[k] = v
        sys.modules['__main__'].__dict__[k] = v

print(f'✓ Exported: {", ".join(imported)}')

# ── إنشاء الـ engine ──────────────────────────────────────────
print('[ENGINE] Initialising CinematicEngineV16…')
engine = CinematicEngineV16()

# تصدير engine للـ global scope
globals()['engine'] = engine
sys.modules['__main__'].__dict__['engine'] = engine

print('[ENGINE] ✓ Ready — models not loaded yet')
print('[ENGINE] شغّل Cell 3 عشان تبدأ الـ WebSocket Bridge')
print(f'[ENGINE] GPU: {torch.cuda.get_device_name(0)}')
print(f'[ENGINE] VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3 — تشغيل الـ WebSocket Bridge                    ║
# ║  الـ cell ده بيفضل شغّال — اضغط ⬛ لو عايز توقفه        ║
# ╚══════════════════════════════════════════════════════════╝

import importlib.util, sys, os

# ── إعدادات ───────────────────────────────────────────────────
# لو بتفتح الـ Dashboard من موبايل أو جهاز تاني غير الـ Colab
# غيّر USE_NGROK لـ True واحط token بتاعك من ngrok.com
USE_NGROK   = False
NGROK_TOKEN = ''     # من https://dashboard.ngrok.com/auth

# ── التأكد من الـ working directory ───────────────────────────
REPO_DIR = '/content/cinematic-engine'
if os.getcwd() != REPO_DIR:
    os.chdir(REPO_DIR)

BRIDGE_PATH = os.path.join(REPO_DIR, 'cev17_backend.py')

if not os.path.exists(BRIDGE_PATH):
    raise FileNotFoundError(
        f'✗ Bridge file not found: {BRIDGE_PATH}\n'
        f'  تأكد إن Cell 1 اشتغل أولاً'
    )

# ── التأكد إن الـ engine شغّال ────────────────────────────────
if 'engine' not in globals() and 'engine' not in sys.modules.get('__main__', {}).__dict__:
    raise RuntimeError(
        '✗ Engine مش محمّل — شغّل Cell 2 الأول'
    )

# ── تحميل الـ bridge ──────────────────────────────────────────
spec   = importlib.util.spec_from_file_location('bridge', BRIDGE_PATH)
bridge = importlib.util.module_from_spec(spec)
sys.modules['bridge'] = bridge
spec.loader.exec_module(bridge)

# ── إعداد ngrok لو مفعّل ─────────────────────────────────────
if USE_NGROK:
    if not NGROK_TOKEN:
        print('⚠ USE_NGROK = True لكن NGROK_TOKEN فاضي')
        print('  روح على https://dashboard.ngrok.com/auth وحط الـ token')
    else:
        from pyngrok import conf
        conf.get_default().auth_token = NGROK_TOKEN
        print('✓ ngrok token configured')

# ── معلومات الاتصال ───────────────────────────────────────────
print('═'*55)
print('  🚀 CINEMATIC ENGINE V17 BRIDGE')
print('═'*55)
print('  افتح الـ Dashboard في المتصفح:')
print('  cinematic_engine_v17_2_final.html')
print()
if USE_NGROK:
    print('  ✓ ngrok مفعّل — الـ URL العام سيظهر بعد ثانية')
    print('  ⚠ غيّر WS_URL في الـ Dashboard للـ URL الجديد')
else:
    print('  WS URL: ws://localhost:7860/ws')
    print('  (يشتغل لو الـ Dashboard والـ Colab على نفس الجهاز)')
print('═'*55)
print('  اضغط ⬛ في أي وقت عشان توقف الـ Bridge')
print('═'*55)
print()

# ── تشغيل الـ bridge ─────────────────────────────────────────
_engine = globals().get('engine') or sys.modules.get('__main__', {}).__dict__.get('engine')
bridge.run_bridge(
    engine=_engine,
    use_ngrok=USE_NGROK
)

# الـ cell ده بيفضل شغّال لحد ما تضغط ⬛

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 4 — اختياري: توليد صورة مباشرة بدون Dashboard    ║
# ║  شغّله بعد Cell 2 — مش محتاج Cell 3                    ║
# ╚══════════════════════════════════════════════════════════╝

# تحميل FLUX SCHNELL (الأسرع والأخف)
engine.load_models(model_mode=ModelMode.FLUX_SCHNELL)

# توليد مشهد سينمائي
cfg = GenerationConfig(
    scene_text = 'A weary detective stands in a rain-soaked alley, neon lights reflecting in puddles',
    camera     = 'Low Angle',
    lighting   = 'Neon Cyberpunk',
    style      = 'Cinematic',
)

print('🎬 Generating scene…')
img, meta = engine.generate_scene(cfg)
print(f'✓ Generated in {meta.get("gen_time", "?"):.1f}s')
img  # بيعرض الصورة مباشرة في الـ notebook